# 01 — Profundidade óptica de neutrinos na matéria (DIS)

**HADROS Sandbox — um conceito por notebook.**

### O que este notebook faz

Responde a **uma** pergunta: *um neutrino de energia $E$ atravessa uma certa quantidade
de matéria — qual é a probabilidade de ele interagir?*

A resposta é a **profundidade óptica** $\tau$, e tudo o que fazemos aqui é montar,
explicar e calcular a integral

$$\boxed{\;\tau(E) \;=\; \int_{\text{caminho}} n_b(\ell)\,\sigma_{\nu N}(E)\,\mathrm{d}\ell\;}
\qquad\qquad P_{\rm int} = 1 - e^{-\tau}$$

### Roteiro

| § | Assunto |
|---|---------|
| 1 | De onde vem a integral (dedução em 5 linhas) |
| 2 | A seção de choque $\sigma_{\nu N}(E)$ do DIS, exatamente como o HADROS3 usa |
| 3 | Caso trivial: densidade constante (a Terra fica opaca a ~40 TeV) |
| 4 | A integral de verdade: meio não-uniforme, discretização e convergência |
| 5 | O meio do HADROS3: o toro analítico em torno do buraco negro |
| 6 | O que o HADROS3 acrescenta por cima disto (Relatividade Geral) |
| 7 | Exercícios |

### O que este notebook **não** faz

Não sorteia *onde* a interação acontece (CDF inversa), não gera o estado final
hadrônico, não propaga geodésicas de Kerr. Cada um desses é um notebook próprio.

### Correspondência com o HADROS3

Toda fórmula aqui é reimplementada do zero, em poucas linhas, mas é **a mesma**
usada em produção. As referências ao código real aparecem em comentários assim:

    # HADROS3: hadros3/dis_sampler.py:1322  ->  d_tau = n_baryon * sigma * comoving_length_rg * r_g_cm


---
## 1. De onde vem a integral

Um neutrino atravessa um meio feito de núcleons (prótons e nêutrons) com
**densidade numérica** $n_b(\ell)$ [cm$^{-3}$], onde $\ell$ é a distância percorrida.

Cada núcleon oferece ao neutrino uma área efetiva $\sigma_{\nu N}$ [cm$^2$] — a seção
de choque. Numa fatia fina de espessura $\mathrm{d}\ell$ e área $A$ há
$n_b A\,\mathrm{d}\ell$ núcleons, que cobrem uma área total $n_b A \sigma\,\mathrm{d}\ell$.
A fração da fatia que é "alvo" é portanto

$$\mathrm{d}P = \frac{n_b A \sigma\,\mathrm{d}\ell}{A} = n_b\,\sigma\,\mathrm{d}\ell .$$

A fatia é fina o suficiente para que os núcleons não se sombreiem — é por isso que
$\mathrm{d}P$ é linear em $\mathrm{d}\ell$. Seja $P_{\rm sob}(\ell)$ a probabilidade de o
neutrino **sobreviver** (não interagir) até $\ell$:

$$\frac{\mathrm{d}P_{\rm sob}}{\mathrm{d}\ell} = -\,n_b(\ell)\,\sigma\,P_{\rm sob}
\quad\Longrightarrow\quad
P_{\rm sob}(\ell) = \exp\!\left[-\int_0^{\ell} n_b(\ell')\,\sigma\,\mathrm{d}\ell'\right]
\equiv e^{-\tau(\ell)} .$$

O expoente é a **profundidade óptica**. É adimensional:

$$[\tau] = \underbrace{\text{cm}^{-3}}_{n_b} \times \underbrace{\text{cm}^{2}}_{\sigma}
\times \underbrace{\text{cm}}_{\mathrm{d}\ell} = 1 .$$

**Três leituras equivalentes de $\tau$:**

1. **Número de livres caminhos médios.** O livre caminho médio é
   $\lambda = 1/(n_b\sigma)$, e $\tau = \ell/\lambda$ num meio uniforme.
2. **Coluna de matéria.** Se a densidade de massa é $\rho$ e a massa do núcleon é
   $m_b$, então $n_b = \rho/m_b$ e

   $$\tau = \frac{\sigma}{m_b}\underbrace{\int \rho\,\mathrm{d}\ell}_{\textstyle X\ [\mathrm{g\,cm^{-2}}]} .$$

   $X$ é a **coluna** (*column depth*), e é o que a geometria do problema
   fornece. Toda a física de neutrinos entra por $\sigma(E)$; toda a geometria e
   astrofísica entram por $X$. Elas só se encontram no produto.
3. **Número médio de interações.** Num processo de Poisson, $\tau$ é o número
   esperado de interações; $P_{\rm int} = 1 - e^{-\tau}$ é a probabilidade de
   haver ao menos uma.

**Regimes:** $\tau \ll 1$ o meio é transparente e $P_{\rm int}\simeq\tau$;
$\tau \gg 1$ o meio é opaco e $P_{\rm int}\to 1$. A fronteira $\tau = 1$ é o
"horizonte" do problema.

> ⚠️ **A hipótese escondida.** $\sigma$ saiu de dentro da integral acima como se fosse
> constante. Ela não é: $\sigma = \sigma(E)$, e $E$ é a energia do neutrino **no
> referencial do meio**. Num meio em repouso e sem gravidade, $E$ é constante ao longo
> do caminho e tudo bem. Perto de um buraco negro, não — é exatamente esse o cuidado
> que o HADROS3 toma, e o assunto do §6.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- constantes, em CGS, iguais às do HADROS3 (hadros3/dis_sampler.py:27-30) ---
M_BARYON_G = 1.67262192369e-24   # massa do núcleon [g]
G_CGS      = 6.67430e-8          # [cm^3 g^-1 s^-2]
C_CGS      = 2.99792458e10       # [cm s^-1]
MSUN_G     = 1.98847e33          # [g]

# --- localiza data/sigma/ subindo a partir do diretório atual -----------------
def find_data_dir(name="sigma"):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "data" / name
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("não achei data/sigma/ — rode o notebook dentro de HADROS3/sandbox")

DATA_SIGMA = find_data_dir()
print("tabelas de seção de choque em:", DATA_SIGMA)

# --- estilo dos gráficos (paleta categórica fixa, na ordem; nunca reciclada) --
SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # azul, laranja, aqua, amarelo
TINTA, TINTA2 = "#1f2328", "#5b6472"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#c9ced6", "axes.labelcolor": TINTA, "axes.titlesize": 11,
    "axes.titleweight": "600", "axes.grid": True, "grid.color": "#e6e9ee",
    "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2.0, "figure.dpi": 110,
})


---
## 2. A seção de choque $\sigma_{\nu N}(E)$ — DIS carregado

O processo é **espalhamento profundamente inelástico** (*deep inelastic
scattering*, DIS) de corrente carregada:

$$\nu_\ell + N \;\longrightarrow\; \ell^- + X$$

O neutrino troca um $W^\pm$ com um quark dentro do núcleon, vira um lépton
carregado, e o núcleon se estilhaça num sistema hadrônico $X$. A seção de choque
duplo-diferencial nas variáveis de Bjorken $x$ (fração de momento do párton) e
$y$ (inelasticidade) é

$$\frac{\mathrm{d}^2\sigma^{\rm CC}}{\mathrm{d}x\,\mathrm{d}y} =
\frac{G_F^2\,s}{4\pi}\left(\frac{m_W^2}{m_W^2+Q^2}\right)^{2}
\Big[\,Y_+ F_2(x,Q^2) - y^2 F_L(x,Q^2) + Y_- \,xF_3(x,Q^2)\,\Big],
\qquad Y_\pm = 1 \pm (1-y)^2,$$

com $s \simeq 2 m_N E_\nu$ e $Q^2 = x y s$. A seção de choque total é a integral
disso em $x$ e $y$ — é ela que precisamos, e é ela que **o HADROS3 lê de tabela**:

$$\sigma_{\nu N}(E) = \int_0^1\!\!\int_0^1 \frac{\mathrm{d}^2\sigma^{\rm CC}}{\mathrm{d}x\,\mathrm{d}y}\,\mathrm{d}x\,\mathrm{d}y .$$

Em energias ultra-altas o $x$ relevante é minúsculo ($x \sim m_W^2/s \lesssim 10^{-6}$),
região onde as PDFs medidas em aceleradores não existem e é preciso um modelo de
**dipolo de cor**. Daí os dois modelos tabelados no HADROS3:

| Modelo | Ideia |
|--------|-------|
| **GBW** | Golec-Biernat–Wüsthoff: dipolo com saturação, escala $Q_s^2(x)$ |
| **IIM** | Iancu–Itakura–Munier: dipolo com saturação melhorada (BK/CGC) |

As tabelas estão em `data/sigma/sigma_nuN_CC_{GBW,IIM}.dat`, com colunas
`E_GeV`, `sigma_GeV^-2`, `sigma_cm2`, cobrindo $10^3$–$10^{14}$ GeV.
O HADROS3 usa a terceira coluna e **interpola em log-log**
(`hadros3/dis_sampler.py:87-98`):

$$\ln\sigma(E) = \ln\sigma_0 + \frac{\ln E - \ln E_0}{\ln E_1 - \ln E_0}\,(\ln\sigma_1 - \ln\sigma_0)$$

Log-log porque $\sigma \propto E^{p}$ aproximadamente — em log-log isso é uma reta,
então a interpolação linear é quase exata. Fora do intervalo tabelado o HADROS3
**recusa** o ponto (levanta erro) em vez de extrapolar; no cálculo da profundidade
óptica esse segmento contribui com $\Delta\tau = 0$.


In [ ]:
def load_sigma_table(model):
    """Lê data/sigma/sigma_nuN_CC_<model>.dat -> (E[GeV], sigma[cm^2]).

    Mesmo parsing de HADROS3 SigmaNuNProvider._load_table (dis_sampler.py:64-85):
    ignora comentários, usa coluna 0 (energia) e coluna 2 (sigma em cm^2),
    e exige grade de energia estritamente crescente.
    """
    path = DATA_SIGMA / f"sigma_nuN_CC_{model}.dat"
    linhas = [l.split() for l in path.read_text().splitlines()
              if l.strip() and not l.startswith("#")]
    E     = np.array([float(p[0]) for p in linhas])
    sigma = np.array([float(p[2]) for p in linhas])
    ok = (E > 0) & (sigma > 0)
    E, sigma = E[ok], sigma[ok]
    assert np.all(np.diff(E) > 0), "grade de energia deve ser crescente"
    return E, sigma


def sigma_cm2(E_gev, tabela):
    """Interpolação log-log da seção de choque. Espelha dis_sampler.py:87-98.

    Fora do intervalo tabelado -> ValueError (o HADROS3 faz o mesmo).
    """
    E_grid, s_grid = tabela
    E = np.atleast_1d(np.asarray(E_gev, dtype=float))
    if E.min() < E_grid[0] or E.max() > E_grid[-1]:
        raise ValueError(f"energia fora da tabela [{E_grid[0]:.3g}, {E_grid[-1]:.3g}] GeV")
    # a interpolação linear em (ln E, ln sigma) É a fórmula log-log do HADROS3
    s = np.exp(np.interp(np.log(E), np.log(E_grid), np.log(s_grid)))
    return s if np.ndim(E_gev) else float(s[0])


TAB = {m: load_sigma_table(m) for m in ("GBW", "IIM")}
for m, (E, s) in TAB.items():
    print(f"{m}: {len(E)} pontos, E = {E[0]:.3g} .. {E[-1]:.3g} GeV, "
          f"sigma = {s[0]:.3e} .. {s[-1]:.3e} cm^2")


### 2.1 Verificação: a interpolação faz o que dissemos?

Dois testes baratos que **você deve sempre fazer** ao reimplementar algo:
reproduzir a fórmula na mão num ponto, e conferir que nos nós da tabela a
interpolação devolve exatamente o valor tabelado.


In [ ]:
E_grid, s_grid = TAB["GBW"]

# (a) nos nós, interpolação == tabela (até erro de arredondamento)
nos = np.array([E_grid[0], E_grid[137], E_grid[-1]])
assert np.allclose(sigma_cm2(nos, TAB["GBW"]), np.interp(nos, E_grid, s_grid), rtol=1e-12)

# (b) num ponto qualquer: fórmula log-log escrita à mão vs. nossa função
i  = 137
E0, E1 = E_grid[i], E_grid[i + 1]
s0, s1 = s_grid[i], s_grid[i + 1]
E_teste = np.sqrt(E0 * E1)                    # meio geométrico do intervalo
t = (np.log(E_teste) - np.log(E0)) / (np.log(E1) - np.log(E0))
manual = np.exp(np.log(s0) + t * (np.log(s1) - np.log(s0)))

print(f"intervalo:  E in [{E0:.5g}, {E1:.5g}] GeV")
print(f"na mão:     sigma = {manual:.6e} cm^2")
print(f"nossa fn:   sigma = {sigma_cm2(E_teste, TAB['GBW']):.6e} cm^2")
assert np.isclose(manual, sigma_cm2(E_teste, TAB["GBW"]), rtol=1e-12)

# (c) fora do intervalo tem de falhar, como no HADROS3
try:
    sigma_cm2(1.0, TAB["GBW"])
except ValueError as e:
    print("fora da tabela -> ValueError:", e)


### 2.2 Escala de referência

Uma tabela de números só é confiável se você tiver com o que compará-la. A
referência clássica é a parametrização de **Gandhi, Quigg, Reno & Sarcevic (1998)**,
ajustada a cálculos DIS com PDFs padrão:

$$\sigma^{\rm CC}_{\nu N}(E) \simeq 5.53\times10^{-36}\ \mathrm{cm^2}\,
\left(\frac{E}{\rm GeV}\right)^{0.363},\qquad 10^{4} < E/\mathrm{GeV} < 10^{12}.$$

Ordens de grandeza para memorizar: $\sigma \sim 10^{-35}$ cm² em 1 TeV,
$\sim 10^{-33}$ cm² em 1 PeV, $\sim 10^{-32}$ cm² em 1 EeV. O expoente $\approx 0.36$,
bem menor que 1, é consequência da saturação: a seção de choque cresce com a
energia, mas devagar.


In [ ]:
def sigma_gqrs(E_gev):
    """Gandhi, Quigg, Reno & Sarcevic (1998), sigma CC nu-N [cm^2]. Só referência."""
    return 5.53e-36 * np.asarray(E_gev, dtype=float) ** 0.363


E = np.logspace(3, 14, 400)

fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.loglog(E, sigma_cm2(E, TAB["GBW"]), color=SERIE[0], label="GBW (HADROS3)")
ax.loglog(E, sigma_cm2(E, TAB["IIM"]), color=SERIE[1], label="IIM (HADROS3)")
ax.loglog(E, sigma_gqrs(E), color=SERIE[2], ls="--", label="GQRS 1998 (referência)")
ax.set_xlabel(r"$E_\nu$  [GeV]")
ax.set_ylabel(r"$\sigma_{\nu N}^{\rm CC}$  [cm$^2$]")
ax.set_title("Seção de choque DIS de corrente carregada")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

print(f"{'E [GeV]':>10} | {'GBW [cm2]':>11} | {'IIM [cm2]':>11} | {'GQRS [cm2]':>11} | GBW/GQRS")
for e in (1e4, 1e6, 1e8, 1e10, 1e12):
    g, i_, q = sigma_cm2(e, TAB["GBW"]), sigma_cm2(e, TAB["IIM"]), sigma_gqrs(e)
    print(f"{e:>10.0e} | {g:>11.3e} | {i_:>11.3e} | {q:>11.3e} | {g/q:>8.1f}")


> 🔎 **Olhe para a última coluna.** As tabelas do HADROS3 ficam **acima** da
> referência GQRS: o GBW por um fator ~7 em $10^4$ GeV e até ~90 em $10^6$ GeV; o IIM
> por um fator ~1–13 dependendo da energia. Isso é grande demais para ser diferença de
> modelo de dipolo.
>
> Não conclua nada a partir daqui — conclua que **isto precisa ser investigado**:
> pode ser normalização por nucleon vs. por núcleo, CC+NC somados, uma unidade
> trocada na geração das tabelas, ou realmente o modelo. O ponto pedagógico é o
> método: *sempre* compare uma tabela numérica com uma parametrização independente
> antes de confiar nela. Como $\tau \propto \sigma$, um fator 90 aqui é um fator 90
> na profundidade óptica.


---
## 3. Caso trivial: densidade constante

Se $n_b$ e $\sigma$ são constantes, a integral é uma multiplicação:

$$\tau = n_b\,\sigma\,L = \frac{\rho}{m_b}\,\sigma\,L .$$

É literalmente isto que está em `hadros3/dis_sampler.py:210`:

```python
def constant_density_tau(n_baryon_cm3, sigma_cm2, length_cm):
    return n_baryon_cm3 * sigma_cm2 * length_cm
```

Vamos usar esse caso para calibrar a intuição com um número que você provavelmente
já ouviu: **a Terra fica opaca a neutrinos em torno de algumas dezenas de TeV.**


In [ ]:
def tau_densidade_constante(rho_g_cm3, comprimento_cm, sigma):
    """tau = (rho/m_b) * sigma * L.  Espelha dis_sampler.py:210."""
    n_b = rho_g_cm3 / M_BARYON_G
    return n_b * sigma * comprimento_cm


def livre_caminho_medio_cm(rho_g_cm3, sigma):
    return M_BARYON_G / (rho_g_cm3 * sigma)


CENARIOS = {   # nome: (rho [g/cm^3], comprimento [cm])
    "1 km de água":        (1.00, 1.0e5),
    "Terra (diâmetro)":    (5.51, 1.2742e9),
    "Sol (diâmetro)":      (1.41, 1.392e11),
}

print(f"{'cenário':<20} {'coluna X [g/cm2]':>17} {'E [GeV]':>10} {'tau':>11} {'P_int':>10}")
for nome, (rho, L) in CENARIOS.items():
    X = rho * L
    for e in (1e4, 1e6, 1e9):
        t = tau_densidade_constante(rho, L, sigma_gqrs(e))
        print(f"{nome:<20} {X:>17.3e} {e:>10.0e} {t:>11.3e} {1-np.exp(-t):>10.4f}")


In [ ]:
# Em que energia a Terra fica opaca (tau = 1), usando a referência GQRS?
rho_terra, L_terra = CENARIOS["Terra (diâmetro)"]
E = np.logspace(3, 12, 2000)
tau_terra = tau_densidade_constante(rho_terra, L_terra, sigma_gqrs(E))
E_opaca = np.interp(1.0, tau_terra, E)          # tau é monótona crescente em E

# conferência analítica: tau = 1  <=>  sigma = m_b / (rho L)
sigma_critica = M_BARYON_G / (rho_terra * L_terra)
E_opaca_analitica = (sigma_critica / 5.53e-36) ** (1 / 0.363)
print(f"sigma crítica          = {sigma_critica:.3e} cm^2")
print(f"E(tau=1) numérica      = {E_opaca:.3e} GeV")
print(f"E(tau=1) analítica     = {E_opaca_analitica:.3e} GeV")
print(f"livre caminho médio em 1 PeV = {livre_caminho_medio_cm(rho_terra, sigma_gqrs(1e6))/1e5:,.0f} km"
      f"  (raio da Terra = 6371 km)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.2))
ax1.loglog(E, tau_terra, color=SERIE[0])
ax1.axhline(1.0, color=TINTA2, lw=1, ls=":")
ax1.axvline(E_opaca, color=TINTA2, lw=1, ls=":")
ax1.annotate(rf"$\tau=1$ em {E_opaca/1e3:.0f} TeV",
             xy=(E_opaca, 1.0), xycoords="data",
             xytext=(0.46, 0.16), textcoords="axes fraction",
             color=TINTA, fontsize=9,
             arrowprops=dict(arrowstyle="-", color=TINTA2, lw=0.9))
ax1.set_xlabel(r"$E_\nu$  [GeV]"); ax1.set_ylabel(r"$\tau$")
ax1.set_title("Terra ao longo do diâmetro: profundidade óptica")

ax2.semilogx(E, 1 - np.exp(-tau_terra), color=SERIE[0])
ax2.axvline(E_opaca, color=TINTA2, lw=1, ls=":")
ax2.set_ylim(-0.02, 1.02)
ax2.set_xlabel(r"$E_\nu$  [GeV]"); ax2.set_ylabel(r"$P_{\rm int} = 1-e^{-\tau}$")
ax2.set_title("... e probabilidade de interagir")
fig.tight_layout()
plt.show()


> ✅ **Sanidade.** $E(\tau=1)$ de algumas dezenas de TeV para a Terra é o número
> conhecido da literatura de IceCube — é por isso que o céu em neutrinos acima de
> ~100 TeV fica visivelmente "vazio" na direção do nadir. Se a sua implementação
> reproduz essa ordem de grandeza, a parte de profundidade óptica está certa.
> (O valor exato depende do perfil de densidade adotado; aqui usamos a densidade
> **média** da Terra, o que subestima um pouco a coluna real ao longo do diâmetro,
> que passa pelo núcleo de ferro.)
>
> Note também os dois gráficos separados. $\tau$ e $P_{\rm int}$ têm escalas
> completamente diferentes — juntá-los num só painel com dois eixos $y$ é a maneira
> mais rápida de enganar a si mesmo.


---
## 4. A integral de verdade: meio não-uniforme

No caso interessante $n_b$ varia ao longo do caminho, e é preciso integrar:

$$\tau = \int_0^L n_b(\ell)\,\sigma(E)\,\mathrm{d}\ell .$$

O HADROS3 já recebe o caminho **quebrado em segmentos** (vêm do integrador de
geodésicas), e soma a contribuição de cada um — `hadros3/dis_sampler.py:1305-1324`:

$$\tau = \sum_i \Delta\tau_i,
\qquad
\Delta\tau_i = \underbrace{\frac{\rho(r_i,\theta_i)}{m_b}}_{n_{b,i}}\;
\sigma(E_i)\;\Delta\ell_i$$

com $\rho$ e $E$ avaliados **no ponto médio** do segmento. Isso é a
**regra do ponto médio**, cujo erro é $O(\Delta\ell^2)$ por segmento e
$O(\Delta\ell^2)$ no total — de segunda ordem, apesar de parecer grosseira.

Antes de aplicar isso ao meio do HADROS3, vamos testar o integrador num caso com
**resposta analítica exata**: um perfil gaussiano

$$\rho(\ell) = \rho_0\,e^{-\ell^2/2w^2}
\qquad\Longrightarrow\qquad
X = \int_{-\infty}^{+\infty}\!\rho\,\mathrm{d}\ell = \rho_0\,w\sqrt{2\pi}.$$

Escolhemos gaussiano de propósito: é a forma do perfil do toro do HADROS3 (§5).


In [ ]:
def tau_por_segmentos(rho_de_ell, L, N, sigma, *, regra="ponto_medio"):
    """Discretiza [0, L] em N segmentos e acumula tau = sum_i n_i * sigma * dl_i.

    É a mesma estrutura do laço de hadros3/dis_sampler.py:1305-1324, só que
    aqui os segmentos são retos e uniformes em vez de virem de uma geodésica.
    """
    bordas = np.linspace(0.0, L, N + 1)
    dl = bordas[1] - bordas[0]
    if regra == "ponto_medio":
        rho = rho_de_ell(0.5 * (bordas[:-1] + bordas[1:]))
        d_tau = (rho / M_BARYON_G) * sigma * dl
    elif regra == "trapezio":
        rho = rho_de_ell(bordas)
        d_tau = 0.5 * (rho[:-1] + rho[1:]) / M_BARYON_G * sigma * dl
    else:
        raise ValueError(regra)
    return float(np.sum(d_tau)), d_tau          # total e as contribuições por segmento


# --- caso-teste com resposta exata -------------------------------------------
from math import erf

rho0, w, L = 3.0, 2.0e5, 2.0e6        # g/cm^3, cm, cm  (integramos de 0 a L, pico em L/2)
sigma_teste = 1.0e-33

rho_gauss = lambda l: rho0 * np.exp(-0.5 * ((l - L / 2) / w) ** 2)

# integral EXATA no domínio finito [0, L] — não a gaussiana infinita.  Usar a
# infinita introduziria um erro de truncamento fixo (~6e-7 aqui) que mascararia
# a convergência do integrador. Compare sempre com o que você realmente integra.
X_exato   = rho0 * w * np.sqrt(2 * np.pi) * erf((L / 2) / (w * np.sqrt(2)))
tau_exato = X_exato / M_BARYON_G * sigma_teste

print(f"gaussiana infinita   X = {rho0 * w * np.sqrt(2*np.pi):.6e} g/cm^2")
print(f"truncada em [0, L]   X = {X_exato:.6e} g/cm^2   "
      f"(cauda cortada = {1 - erf((L/2)/(w*np.sqrt(2))):.2e})")

print(f"tau exato            = {tau_exato:.6e}\n")
for N in (4, 16, 64, 256):
    t_m, _ = tau_por_segmentos(rho_gauss, L, N, sigma_teste, regra="ponto_medio")
    t_t, _ = tau_por_segmentos(rho_gauss, L, N, sigma_teste, regra="trapezio")
    print(f"N={N:>4}  ponto médio {t_m:.6e} (err {abs(t_m/tau_exato-1):.2e})"
          f"   trapézio {t_t:.6e} (err {abs(t_t/tau_exato-1):.2e})")


In [ ]:
Ns = np.unique(np.logspace(0.5, 3.5, 22).astype(int))
err_m = [abs(tau_por_segmentos(rho_gauss, L, N, sigma_teste, regra="ponto_medio")[0] / tau_exato - 1) for N in Ns]
err_t = [abs(tau_por_segmentos(rho_gauss, L, N, sigma_teste, regra="trapezio")[0] / tau_exato - 1) for N in Ns]

fig, ax = plt.subplots(figsize=(6.8, 4.4))
ax.loglog(Ns, err_m, "o-", color=SERIE[0], ms=5, label="ponto médio (HADROS3)")
ax.loglog(Ns, err_t, "s-", color=SERIE[1], ms=5, label="trapézio")
# reta de referência ANCORADA nos dados (senão a comparação de inclinações
# vira chute visual): passa pelo ponto de N mais próximo de 100
j_anc = int(np.argmin(abs(Ns - 100)))
ax.loglog(Ns, err_m[j_anc] * (np.asarray(Ns, float) / Ns[j_anc]) ** -2.0,
          ls=":", color=TINTA2, lw=1.4, label=r"referência $\propto N^{-2}$")

inclinacao = np.polyfit(np.log(Ns[Ns >= 60]), np.log(np.asarray(err_m)[Ns >= 60]), 1)[0]
print(f"inclinação medida (N >= 60), regra do ponto médio: {inclinacao:.3f}  (esperado -2)")
ax.set_xlabel("número de segmentos $N$")
ax.set_ylabel(r"erro relativo em $\tau$")
ax.set_title("Convergência do integrador: as duas regras são de 2ª ordem")
ax.legend()
fig.tight_layout()
plt.show()


> No regime assintótico (aqui $N \gtrsim 60$; a inclinação medida está impressa
> acima) as duas curvas descem com inclinação $-2$ em log-log: dobrar o número de
> segmentos divide o erro por 4. É por isso que a regra do ponto médio, que parece
> grosseira, é perfeitamente adequada — o que domina o custo não é a ordem do
> método, é o número de segmentos que a geodésica fornece.
>
> Repare no comentário do código sobre a integral exata: se compararmos com a
> gaussiana **infinita** em vez da truncada em $[0,L]$, a curva de erro trava num
> patamar de $\sim 6\times10^{-7}$ e parece que o integrador parou de convergir.
> Não parou — o que travou foi a *referência*. Errar o alvo é um jeito clássico de
> "descobrir" um bug que não existe.
>
> **Consequência prática para o HADROS3:** a resolução da profundidade óptica é
> controlada pelo passo do integrador de geodésicas. Se você suspeitar de um valor
> de $\tau$, o primeiro teste é refazer com o passo pela metade e ver se muda — se
> mudar, é discretização, não física.


---
## 5. O meio do HADROS3: o toro analítico

O HADROS3 modela a matéria em torno do buraco negro como um **toro analítico**
(`hadros3/medium_model.py:25-47`): corte radial duro entre $r_{\rm in}$ e $r_{\rm out}$,
e perfis gaussianos em $r$ e em $\theta$:

$$\rho(r,\theta) = \rho_0 \;
\exp\!\left[-\tfrac{1}{2}\Big(\tfrac{r-r_{\rm pico}}{\sigma_r}\Big)^{2}\right]
\exp\!\left[-\tfrac{1}{2}\Big(\tfrac{\theta-\pi/2}{\sigma_\theta}\Big)^{2}\right],
\qquad r_{\rm in}\le r \le r_{\rm out},$$

$$\sigma_r = \tfrac{1}{2}(r_{\rm out}-r_{\rm in}), \qquad \sigma_\theta = \theta_{\rm meia-abertura}.$$

⚠️ Um detalhe que engana muita gente: **o ângulo de meia-abertura não é uma
borda.** É a largura da gaussiana. O toro não "acaba" em $\theta_{\rm meia-abertura}$;
ele só fica exponencialmente rarefeito. Existe densidade no eixo polar, pequena mas
não nula — e como veremos, isso é justamente o que decide por onde os neutrinos
escapam.

Distâncias vêm em $r_g = GM/c^2$; a conversão para cm é `dis_sampler.py:177-178`.
Usamos aqui a configuração default do HADROS3 (`presets/hadros_web/default_config.json`).


In [ ]:
# ---- configuração default do HADROS3 ----------------------------------------
CFG = {
    "mass_msun":              3.0,     # buraco negro estelar
    "r_inner_rg":             6.0,
    "r_outer_rg":            14.0,
    "r_peak_rg":             10.0,
    "half_opening_angle_deg": 10.0,
    "density_norm_g_cm3":     1.0e10,
}

def rg_to_cm(mass_msun):
    """Raio gravitacional em cm. dis_sampler.py:177-178."""
    return G_CGS * mass_msun * MSUN_G / (C_CGS * C_CGS)


def rho_toro(r_rg, theta_rad, cfg=CFG):
    """Densidade do toro analítico [g/cm^3]. Espelha medium_model.py:25-47."""
    r     = np.asarray(r_rg, dtype=float)
    theta = np.asarray(theta_rad, dtype=float)
    sigma_r     = 0.5 * (cfg["r_outer_rg"] - cfg["r_inner_rg"])
    sigma_theta = np.radians(cfg["half_opening_angle_deg"])
    perfil_r     = np.exp(-0.5 * ((r - cfg["r_peak_rg"]) / sigma_r) ** 2)
    perfil_theta = np.exp(-0.5 * ((theta - 0.5 * np.pi) / sigma_theta) ** 2)
    rho = cfg["density_norm_g_cm3"] * perfil_r * perfil_theta
    return np.where((r < cfg["r_inner_rg"]) | (r > cfg["r_outer_rg"]), 0.0, rho)


R_G_CM = rg_to_cm(CFG["mass_msun"])
print(f"M = {CFG['mass_msun']} Msol  ->  r_g = {R_G_CM:.4e} cm = {R_G_CM/1e5:.2f} km")
print(f"rho no pico (equador) = {float(rho_toro(CFG['r_peak_rg'], np.pi/2)):.3e} g/cm^3")
print(f"rho no pico a 30° do equador = {float(rho_toro(CFG['r_peak_rg'], np.pi/2 + np.radians(30))):.3e} g/cm^3")


In [ ]:
# ---- mapa meridional da densidade + os raios que vamos integrar --------------
lim = 18.0
Rg, Zg = np.meshgrid(np.linspace(0, lim, 420), np.linspace(-lim, lim, 640))
rr     = np.hypot(Rg, Zg)
tt     = np.arctan2(Rg, Zg)                      # theta a partir do eixo z
dens   = rho_toro(rr, tt)

fig, ax = plt.subplots(figsize=(6.4, 6.6))
mapa = ax.pcolormesh(Rg, Zg, np.log10(np.maximum(dens, 1e-8)),
                     cmap="Blues", shading="auto", vmin=-2, vmax=10)
cb = fig.colorbar(mapa, ax=ax, shrink=0.86, pad=0.02)
cb.set_label(r"$\log_{10}\,\rho$  [g cm$^{-3}$]")

ANGULOS_DEG = [0.0, 20.0, 45.0, 70.0]            # graus a partir do equador
for k, a in enumerate(ANGULOS_DEG):
    th = np.pi / 2 - np.radians(a)
    ax.plot([0, lim * np.sin(th)], [0, lim * np.cos(th)],
            color=SERIE[k % len(SERIE)], lw=1.8, label=f"raio a {a:.0f}° do equador")
ax.add_patch(plt.Circle((0, 0), 1.0, color="black"))
ax.set_xlim(0, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
ax.set_xlabel(r"$R = r\sin\theta$  [$r_g$]"); ax.set_ylabel(r"$z = r\cos\theta$  [$r_g$]")
ax.set_title("Toro analítico do HADROS3 (corte meridional)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(False)
fig.tight_layout()
plt.show()


### 5.1 A integral ao longo de um raio

Simplificação deliberada deste notebook: o neutrino sai da origem em **linha reta**
com ângulo polar $\theta$ fixo. (No HADROS3 o caminho é uma geodésica de Kerr — §6.)
Com $\theta$ constante, a integral vira unidimensional em $r$:

$$\tau(\theta, E) = \frac{\sigma(E)}{m_b}\; r_g \int_{r_{\rm in}}^{r_{\rm out}}
\rho(r,\theta)\,\mathrm{d}r
\;=\;
\underbrace{\frac{\sigma(E)}{m_b}\,r_g \int_{r_{\rm in}}^{r_{\rm out}}\!\!\rho(r,\tfrac{\pi}{2})\,\mathrm{d}r}_{\textstyle \tau_{\rm eq}(E)}
\;\times\;
e^{-\frac{1}{2}\left(\frac{\theta-\pi/2}{\sigma_\theta}\right)^{2}}$$

O fator $\theta$ sai da integral porque não depende de $r$ — logo
$\tau(\theta) = \tau_{\rm eq}\,e^{-\Delta\theta^2/2\sigma_\theta^2}$ **exatamente**.
Isso nos dá uma verificação analítica de graça para o resultado numérico.

O fator $r_g$ aparece porque integramos $r$ em unidades de $r_g$ e $\sigma$ está em
cm² — exatamente o `* r_g_cm` de `dis_sampler.py:1322`.


In [ ]:
def tau_raio(theta_rad, E_gev, modelo="GBW", N=400, cfg=CFG):
    """tau ao longo de um raio radial reto no ângulo polar theta.

    Devolve (tau_total, r_meio, d_tau_por_segmento) — a decomposição por segmento
    é o análogo de segment_tau_records em dis_sampler.py:1331-1342.
    """
    bordas = np.linspace(cfg["r_inner_rg"], cfg["r_outer_rg"], N + 1)
    dr_rg  = bordas[1] - bordas[0]
    r_meio = 0.5 * (bordas[:-1] + bordas[1:])                     # ponto médio
    rho    = rho_toro(r_meio, theta_rad, cfg)                     # g/cm^3
    n_b    = rho / M_BARYON_G                                     # cm^-3
    sigma  = sigma_cm2(E_gev, TAB[modelo])                        # cm^2
    dl_cm  = dr_rg * rg_to_cm(cfg["mass_msun"])                   # cm
    d_tau  = n_b * sigma * dl_cm                                  # adimensional
    return float(np.sum(d_tau)), r_meio, d_tau


E_demo = 1.0e6                                        # 1 PeV
tau_eq, r_meio, d_tau = tau_raio(np.pi / 2, E_demo)

X_eq = np.sum(rho_toro(r_meio, np.pi/2) * (r_meio[1]-r_meio[0]) * R_G_CM)
print(f"E = {E_demo:.0e} GeV, modelo GBW, raio equatorial")
print(f"  coluna X          = {X_eq:.3e} g/cm^2")
print(f"  sigma             = {sigma_cm2(E_demo, TAB['GBW']):.3e} cm^2")
print(f"  tau               = {tau_eq:.3e}")
print(f"  P_int             = {1 - np.exp(-tau_eq):.6f}")
print(f"  livre caminho médio no pico = {livre_caminho_medio_cm(1e10, sigma_cm2(E_demo, TAB['GBW']))/R_G_CM:.3e} r_g")
print("\n  primeiros segmentos (compare com dis_sampler.py:1331-1342):")
print(f"  {'r_meio [rg]':>12} {'rho [g/cm3]':>13} {'n_b [1/cm3]':>13} {'d_tau':>12}")
for j in range(0, 400, 80):
    print(f"  {r_meio[j]:>12.3f} {float(rho_toro(r_meio[j], np.pi/2)):>13.4e} "
          f"{float(rho_toro(r_meio[j], np.pi/2))/M_BARYON_G:>13.4e} {d_tau[j]:>12.4e}")


In [ ]:
# ---- acúmulo de tau ao longo do raio, para várias energias -------------------
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for k, e in enumerate([1e4, 1e6, 1e9, 1e12]):
    _, r_m, dt = tau_raio(np.pi / 2, e)
    ax.semilogy(r_m, np.cumsum(dt), color=SERIE[k], label=f"E = {e:.0e} GeV")
ax.axhline(1.0, color=TINTA2, lw=1, ls=":")
ax.text(6.2, 1.6, r"$\tau=1$", color=TINTA2, fontsize=9)
ax.set_xlabel(r"$r$  [$r_g$]")
ax.set_ylabel(r"$\tau$ acumulada, $\sum_{i \leq n} \Delta\tau_i$")
ax.set_title("Acúmulo da profundidade óptica ao longo do raio equatorial (GBW)")
ax.legend(loc="center right")
fig.tight_layout()
plt.show()


> O toro do HADROS3 no equador é **absurdamente opaco**: $\tau \sim 10^{9}$ em 1 PeV.
> $\tau$ passa de 1 já nos primeiros $\sim 10^{-8}\,r_g$ de matéria. Nenhum neutrino
> sai por ali — todos interagem quase imediatamente ao entrar no toro.
>
> Isso é uma consequência direta de $\rho_0 = 10^{10}$ g/cm³ (densidade de anã branca)
> ao longo de $\sim 8\,r_g \approx 35$ km. A coluna é $\sim 3\times10^{16}$ g/cm² —
> uns 4 milhões de vezes a coluna da Terra inteira ($7\times10^{9}$ g/cm²).


### 5.2 Onde o meio deixa de ser opaco?

Como o perfil em $\theta$ é gaussiano e não tem borda, existe um ângulo a partir do
qual $\tau$ cai abaixo de 1. Esse ângulo é a "janela" por onde os neutrinos escapam,
e ele é **fortemente dependente da energia**: mais energia → mais $\sigma$ → janela
mais estreita.

Resolvendo $\tau(\theta) = 1$ com a fórmula fatorada da §5.1:

$$\Delta\theta_{\rm crit} = \sigma_\theta\,\sqrt{2\ln \tau_{\rm eq}} .$$


In [ ]:
angulos = np.linspace(0, 89, 400)                    # graus a partir do equador
thetas  = np.pi / 2 - np.radians(angulos)

fig, ax = plt.subplots(figsize=(7.4, 4.8))
for k, e in enumerate([1e4, 1e6, 1e9, 1e12]):
    tau_ang = np.array([tau_raio(t, e)[0] for t in thetas])
    ax.semilogy(angulos, tau_ang, color=SERIE[k], label=f"E = {e:.0e} GeV")

    # verificação analítica: tau(theta) = tau_eq * exp(-dtheta^2 / 2 sigma_theta^2)
    tau_eq_e   = tau_ang[0]
    sig_th_deg = CFG["half_opening_angle_deg"]
    previsto   = tau_eq_e * np.exp(-0.5 * (angulos / sig_th_deg) ** 2)
    assert np.allclose(tau_ang, previsto, rtol=1e-10), "fatoração em theta falhou"

    dtheta_crit = sig_th_deg * np.sqrt(2 * np.log(tau_eq_e))
    ax.plot([dtheta_crit], [1.0], "o", color=SERIE[k], ms=7, zorder=5)
    print(f"E = {e:>8.0e} GeV:  tau_equatorial = {tau_eq_e:.3e},  "
          f"transparente além de {dtheta_crit:.1f}° do equador")

ax.axhline(1.0, color=TINTA2, lw=1, ls=":")
ax.set_xlabel("ângulo a partir do equador  [graus]")
ax.set_ylabel(r"$\tau$")
ax.set_ylim(1e-12, 1e12)
ax.set_title(r"Janela de escape: $\tau$ vs. ângulo (pontos = $\tau=1$)")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


> O `assert` dentro do laço confirma numericamente a fatoração analítica em
> $\theta$ — a integral numérica e a fórmula fechada concordam em 10 dígitos. É esse
> tipo de teste barato que torna um cálculo complicado verificável por um humano.
>
> Fisicamente: a fonte é opaca no plano equatorial e transparente perto dos polos, e a
> fronteira anda com a energia. É por isso que a estrutura angular do meio — e não só
> a densidade total — determina o que um observador distante vê.


---
## 6. O que o HADROS3 acrescenta por cima disto

Tudo acima é a física correta em espaço plano. Perto de um buraco negro de Kerr
três coisas mudam, e o HADROS3 trata as três (`hadros3/dis_sampler.py:185-247`):

**(a) O caminho não é reto.** $\ell$ é o parâmetro ao longo de uma geodésica nula
de Kerr, não uma linha reta. Os segmentos vêm do integrador de geodésicas
(`hadros3/forward_geodesics.py`).

**(b) O comprimento tem de ser o do referencial da matéria.** O que entra na
integral é o comprimento próprio medido *pelo meio*, não uma coordenada. O
HADROS3 usa o invariante

$$\Delta\ell_{\rm com} = -(u\cdot k)\,\Delta\lambda \, ,$$

com $u^\mu$ a quadrivelocidade do meio. Isso é invariante sob reescala recíproca
do momento e do parâmetro afim — a razão de o código dividir pela normalização
`momentum_affine_normalization_gev` (`dis_sampler.py:214-231`).

**(c) A energia é local.** $\sigma$ tem de ser avaliada na energia que o *meio* vê:

$$E_{\rm local} = -u^\mu p_\mu \, .$$

Para um observador estático $u^t = 1/\sqrt{-g_{tt}}$; dentro da ergosfera não
existe observador estático, e o HADROS3 cai para o observador **ZAMO** (*zero
angular momentum observer*), registrando isso como aproximação
(`dis_sampler.py:185-203`).

A integral invariante completa que o código discretiza é (Theory §"Optical depth"):

$$\tau = \int n_b \; \sigma_{\nu N}(-u\!\cdot\!p)\;\frac{-u\!\cdot\!p}{E_{\rm norm}}\;\mathrm{d}\lambda\;r_g .$$

Compare com a integral do §1: é a mesma coisa, escrita de forma covariante.

### Quanto isso importa, numericamente?

$\sigma \propto E^{0.363}$, então um desvio para o azul de fator $f$ na energia local
muda $\tau$ por $f^{0.363}$ — efeito suave. Já o comprimento próprio entra
**linearmente**. A conta abaixo estima as duas sensibilidades.


In [ ]:
print(f"{'fator na energia local':>24} | {'efeito em sigma (=efeito em tau)':>34}")
for f in (0.5, 1.0, 2.0, 5.0, 10.0):
    r = sigma_cm2(1e6 * f, TAB["GBW"]) / sigma_cm2(1e6, TAB["GBW"])
    print(f"{f:>24.1f} | {r:>34.3f}")

print("\nExpoente local d(ln sigma)/d(ln E) do GBW, em torno de 1 PeV:")
e0, e1 = 0.9e6, 1.1e6
p = (np.log(sigma_cm2(e1, TAB["GBW"])) - np.log(sigma_cm2(e0, TAB["GBW"]))) / (np.log(e1) - np.log(e0))
print(f"  p = {p:.3f}   (a referência GQRS tem p = 0.363)")

custo_energia = sigma_cm2(2e6, TAB["GBW"]) / sigma_cm2(1e6, TAB["GBW"]) - 1
print(f"\nMoral: um erro de fator 2 na energia local custa {100*custo_energia:.0f}% em tau;")
print("um erro de fator 2 no comprimento próprio custa 100%. Cuidado com a geometria.")


---
## 7. Exercícios

1. **Unidades.** Refaça o §3 em unidades naturais ($\hbar = c = 1$, energias em GeV),
   usando a segunda coluna das tabelas (`sigma_GeV^-2`). Confira que
   $1\ \mathrm{GeV^{-2}} = 3.894\times10^{-28}\ \mathrm{cm^2}$ e que você recupera os
   mesmos $\tau$.

2. **Modelo importa?** Refaça o §5.2 com `modelo="IIM"`. De quantos graus muda a
   janela de escape em 1 PeV? Explique por que a mudança em graus é tão pequena mesmo
   com $\sigma$ diferindo por uma ordem de grandeza. *(Dica: $\Delta\theta_{\rm crit}
   \propto \sqrt{\ln\tau_{\rm eq}}$.)*

3. **Densidade crítica.** Qual $\rho_0$ tornaria o toro equatorial marginalmente
   opaco ($\tau = 1$) em 1 PeV? Compare com $10^{10}$ g/cm³. O que isso diz sobre a
   escolha default do HADROS3?

4. **Terra de verdade.** Troque a Terra de densidade constante por um perfil PREM
   (2 ou 3 camadas já bastam) e recalcule $\tau$ em função do ângulo de nadir. Compare
   com a curva de constante. Onde a aproximação de densidade constante erra mais?

5. **Convergência no caso real.** Refaça o §5.1 variando `N` de 10 a 10 000. Qual `N`
   dá $\tau$ com 4 dígitos corretos? Por que o toro precisa de mais segmentos que a
   gaussiana do §4?

6. **A discrepância do §2.2.** Investigue: as tabelas do HADROS3 são por núcleon ou
   por núcleo? CC apenas ou CC+NC? Confira a coluna `sigma_GeV^-2` contra a
   `sigma_cm2` — a conversão está certa? *(Comece por `cpp/apps/hadros3_dis_sampler.cpp`
   e pela geração das tabelas.)* Este é um exercício de verdade: a resposta não está
   neste notebook.


---
## Resumo

| Conceito | Fórmula | Onde no HADROS3 |
|---|---|---|
| Profundidade óptica | $\tau = \int n_b \sigma\,\mathrm{d}\ell$ | `dis_sampler.py:1305-1324` |
| Densidade numérica | $n_b = \rho/m_b$ | `dis_sampler.py:1312` |
| Densidade constante | $\tau = n_b\sigma L$ | `dis_sampler.py:210` |
| Seção de choque | log-log interpolada em tabela | `dis_sampler.py:87-98` |
| Densidade do meio | toro gaussiano | `medium_model.py:25-47` |
| Probabilidade | $P = 1-e^{-\tau}$ (via `expm1`) | `dis_sampler.py:206-207` |
| Comprimento próprio | $-(u\cdot k)\,\Delta\lambda$ | `dis_sampler.py:214-231` |
| Energia local | $-u^\mu p_\mu$ (estático ou ZAMO) | `dis_sampler.py:185-203` |

**Próximo notebook:** dado $\tau$, *onde* ao longo do caminho a interação acontece —
amostragem por CDF inversa, $P(i) = \Delta\tau_i / \sum_j \Delta\tau_j$.
